# Práctica de detección de objetos

### **Grupo:** N — **Integrantes:** Lucas Amado Oseira y María Lestón Geada
##### **Dataset:** Lettuce Pallets (`Francesco/lettuce-pallets`)
##### **Detectores:** Cascade R-CNN (Detectron2) y SSD300-VGG16 (Torchvision), ambos preentrenados en MS COCO

## Cómo usar este cuaderno y resumen de decisiones

**Orden de trabajo (se ejecuta de arriba abajo, en Kaggle con GPU y *Internet* activado en los ajustes del notebook):**
1. Instalar dependencias → 2. Configurar → 3. Cargar y limpiar los datos → 4. Analizar el dataset →
5. Infraestructura común → 6. Baseline SSD300 → 7. Baseline Cascade R-CNN → 8. Resultados.

Todo se guarda en `/kaggle/working/work` (checkpoints, predicciones, figuras y `results.csv`). Si un entrenamiento ya existe en esa carpeta,
**se reutiliza** y solo se recalcula la evaluación. Los resultados de todos los experimentos se acumulan en `results.csv`.

**Resumen de decisiones (cada una se justifica en su sección):**

| Decisión | Elección | Motivo breve |
|---|---|---|
| Clases | Se elimina `lettuce` (id 0) → 5 clases | Es la categoría "padre" de Roboflow, sin anotaciones; el guion habla de 5 clases |
| Protocolo | Entrenar en *train*, elegir en *val*, reportar *test* | Evita sobreajustar las decisiones al test |
| Baseline | Fine-tuning de pesos COCO, **sin aumento de datos**, un solo LR | Punto de partida limpio |
| Evaluación | Mismo código (`evaluate_coco`) y mismos umbrales para ambos detectores | Comparación justa SSD vs Cascade |

## 1. Imports y configuración del entorno

Ejecutamos esta celda una vez por sesión de Kaggle (acelerador **GPU** e **Internet** activados en *Settings*). Detectron2 se compila desde el repositorio oficial (tarda varios minutos).
Si la sesión pide reiniciar el kernel, lo reiniciamos y continuamos desde la celda de imports.

In [ ]:
# pycocotools ya viene en Kaggle; por si acaso
!pip -q install pycocotools

# Detectron2 se compila desde el repositorio (tarda varios minutos); se omite si ya está instalado
!python -c "import detectron2" 2>/dev/null || pip -q install 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation

Importamos las librerías necesarias para realizar la práctica.

In [ ]:
import os, sys, io, json, time, math, copy, random, tarfile, contextlib, gc, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from datasets import load_dataset

import torch
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Importamos las librerías de pycocotools para la evaluación del modelo y el manejo del dataset COCO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Importamos las librerías de Detectron2 para la configuración, entrenamiento y evaluación del modelo
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator
from detectron2.data import (DatasetCatalog, MetadataCatalog, DatasetMapper,
                                build_detection_train_loader, transforms as T)
from detectron2.data.datasets import register_coco_instances

La práctica se ejecuta en Kaggle. Definimos las constantes: rutas de trabajo (`/kaggle/working`), el origen del dataset y los parámetros de inferencia comunes a ambos detectores.
Además, fijaremos una semilla que nos garantice la reproducibilidad de los resultados.

**Origen del dataset** (se prueba en este orden): 1) datos ya preparados en `data/raw`; 2) carpetas Roboflow (`train/`, `valid/`, `test/`) o un `dataset.tar.gz` añadidos como *Input* del notebook;
3) descarga desde Hugging Face (`Francesco/lettuce-pallets`, requiere Internet) y exportación al mismo formato Roboflow, de modo que **la limpieza posterior es idéntica en los tres casos**.

In [ ]:
SEED = 2710
DATASET_NAME = "lettuce-pallets"
HF_DATASET = "Francesco/lettuce-pallets"   # solo se usa si no hay dataset en /kaggle/input

IN_KAGGLE = Path("/kaggle/working").exists()
KAGGLE_INPUT = Path("/kaggle/input")
ROOT_DIR = Path("/kaggle/working") if IN_KAGGLE else Path(".")   # en local se usa el directorio actual
WORK_DIR = ROOT_DIR / "work"    # checkpoints, predicciones, figuras y results.csv
DATA_DIR = ROOT_DIR / "data"    # datos extraídos y limpios

PRETRAINED = True # pesos preentrenados en COCO

# Parámetros de inferencia comunes a ambos detectores
SCORE_THRESH_EVAL = 0.001
NMS_IOU = 0.5
MAX_DETS = 300

SPLITS = ("train", "val", "test")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| Kaggle:", IN_KAGGLE)

In [ ]:
# Fijamos todas las semillas para asegurar la reproducibilidad de los experimentos
def seed_everything(seed=SEED, deterministic=True):
    """Fija todas las semillas. 'deterministic' hace cuDNN determinista (algo más lento)."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Función para fijar la semilla de los workers del DataLoader, si no el aumento de datos no es reproducible
def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# Creamos un generador de números aleatorios para el DataLoader, que se utilizará para inicializar los workers y garantizar la reproducibilidad
def make_generator(seed=SEED):
    return torch.Generator().manual_seed(seed)

seed_everything() # Fijamos la semilla

# Creamos los subdirectorios necesarios para almacenar los resultados del entrenamiento y la evaluación
for sub in ("ckpt", "preds", "figs", "d2"):
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)

# Creamos los subdirectorios necesarios para almacenar los datos limpios y procesados del dataset
DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR = DATA_DIR / "clean" # directorio para almacenar los datos limpios y procesados del dataset
CLEAN_DIR.mkdir(exist_ok=True)
print("WORK_DIR:", WORK_DIR.resolve())

## 2. Datos: extracción, limpieza y formato unificado

Comenzaremos a tratar los datos: localizamos el dataset (ver orden de búsqueda más arriba) y el JSON y la carpeta de imágenes de cada split. Localizamos los splits por el nombre exacto de la carpeta (`train`, `valid`, `test`) de forma robusta.

Con la función ``export_hf`` descargamos el dataset desde Hugging face con la librería ``load_dataset``, mapeamos los nombres a las carpetas correspondientes con los splits, según el diccionario ``SPLIT_DIR_ALIASES``. Guardamos cada imagen como 00000.jpg, 00001.jpg... y su id es el índice, de esta forma los nombres de archivo e ids son únicos y no dependemos del campo ``image_id`` en el que podían aparecer repeticiones entre splits.

Añadimos las anotaciones siguiendo el siguiente formato ``bbox = [x, y, w, h]``. x, y: esquina superior izquerda, w: ancho, h: alto. Especialmente útil porque iremos guardando en un diccionario de anotaciones con la siguiente forma:

```python

{"id": n, "image_id": idx, "category_id": int(cat),
 "bbox": [x, y, w, h], "area": w*h, "iscrowd": 0}
```
podemos emplear h y w apra calcular el área que seráempleada por COCO para evaluar como pequeño/mediano/grande. ``id`` identifica dentro del split y ``image_id`` es el índice de la imagen. ``category_id`` se corresponde con ClassLabel, seguimos mantenindo lettuce como id 0, **MERI AO MELLOR QUERES CARGARTE ESTO** será eliminado en la fase de limpieza de los datos. iscrowd = para objetos individuales y no multitudes

In [ ]:
SPLIT_DIR_ALIASES = {"train": "train", "valid": "val", "validation": "val", "val": "val", "test": "test"}

# Extraemos el tar.gz si el destino está vacío
def extract_tar(tar_path, dest):
    """Extrae el tar.gz si el destino está vacío."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if not any(dest.iterdir()):
        with tarfile.open(tar_path, "r:*") as tar:
            try: tar.extractall(dest, filter="data")
            except TypeError: tar.extractall(dest)
    return dest

# Localizamos los splits de entrenamiento, validación y test en el dataset extraído
def locate_raw_splits(root):
    raw = {}
    for js in Path(root).rglob("*.json"):
        key = SPLIT_DIR_ALIASES.get(js.parent.name.lower())
        if key and key not in raw:
            raw[key] = {"json": js, "images": js.parent} # en Roboflow, las imágenes están junto al JSON
    missing = [s for s in SPLITS if s not in raw]
    assert not missing, f"Faltan splits: {missing}"
    return raw

def find_tar_in_input():
    if not KAGGLE_INPUT.exists(): return None
    for pat in ("*.tar.gz", "*.tgz", "*.tar"):
        found = sorted(KAGGLE_INPUT.rglob(pat))
        if found: return found[0]
    return None

def export_hf(dest):
    """Descarga el dataset de Hugging Face y lo escribe como Roboflow: <split>/{imágenes, _annotations.coco.json}.
    Se conservan TODAS las categorías originales (incluida 'lettuce', id 0) para que build_clean_coco haga el mismo trabajo."""

    #Descargamos en memoria el dataset de Hugging Face y lo escribimos en el directorio de destino como Roboflow: <split>/{imágenes, _annotations.coco.json}.
    hf_ds = load_dataset(HF_DATASET)
    # Mapeamos los nombres internos de los splits con  los nombres de directorio que queremos usar en el destino
    dir_names = {"train": "train", "val": "valid", "test": "test"}
    # Accedemos al primer split disponible e inspeccionamos el esquema (features) de categoría de objects para tener los nombres de las clases
    feat = hf_ds[next(iter(hf_ds.keys()))].features["objects"]["category"]
    # En caso de que la columna de categoríais esté eun un contenedor de tipo Sequence/List, obtenemos el feature ClassLabel subyacente
    feat = getattr(feat, "feature", feat)                       # Sequence/List(ClassLabel) -> ClassLabel
    # Si el feature es de tipo ClassLabel, obtenemos los nombres de las clases; si no, dejamos names como None
    names = list(feat.names) if getattr(feat, "names", None) else None
    # Inicializamos un conjunto para almacenar las categorías observadas en el dataset
    observed = set()

    # Recorremos cada split del dataset
    for split_name, ds in hf_ds.items():
        # Normalizamos el nombre del split a minúsculas y determinamos la clave correspondiente en dir_names
        s = split_name.lower()
        key = "train" if s == "train" else "val" if "val" in s else "test" if "test" in s else None
        if key is None: continue
        out = Path(dest) / dir_names[key]
        out.mkdir(parents=True, exist_ok=True)
        # Inicializamos listas para almacenar la información de las imágenes y anotaciones
        images, anns = [], []
        # Iteramos sobre cada fila del split del dataset convirtiendo las imágenes a RGB y guardándolas en el directorio correspondiente, mientras recopilamos la información de las anotaciones
        for idx, row in enumerate(ds):
            img = row["image"].convert("RGB")
            fn = f"{idx:05d}.jpg"; img.save(out / fn, quality=95)
            images.append({"id": idx, "file_name": fn, "width": img.width, "height": img.height})
            objs = row["objects"]
            # Emparejamos cada bounding box con su categoría correspondiente y agregamos la información a la lista de anotaciones, mientras registramos las categorías observadas
            for bbox, cat in zip(objs["bbox"], objs["category"]):
                x, y, w, h = map(float, bbox); observed.add(int(cat))
                anns.append({"id": len(anns) + 1, "image_id": idx, "category_id": int(cat),
                             "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
        # Escribimos el archivo _annotations.coco.json para el split actual, incluyendo las imágenes y anotaciones recopiladas
        (out / "_annotations.coco.json").write_text(json.dumps({"images": images, "annotations": anns, "categories": []}))
    # Actualizamos el archivo _annotations.coco.json para cada split, incluyendo las categorías
    cat_ids = range(len(names)) if names else sorted(observed)
    cats = [{"id": i, "name": names[i] if names else str(i)} for i in cat_ids]
    for d in dir_names.values():
        p = Path(dest) / d / "_annotations.coco.json"
        j = json.loads(p.read_text()); j["categories"] = cats; p.write_text(json.dumps(j))
    return dest

raw_root = DATA_DIR / "raw"
if not raw_root.exists() or not any(raw_root.iterdir()):
    print("Preparando datos RAW...")
    RAW = locate_raw_splits(raw_root)
for s in SPLITS: print(f"{s}: {RAW[s]['json']}")

A continuación, cargamos el dataset de entrenamiento COCO y mostramos información sobre las distintas categorías de lechuga y las anotaciones existentes para cada imagen.

In [ ]:
# Información del dataset ANTES de la limpieza (lee el JSON original de Roboflow)
raw_train = COCO(str(RAW["train"]["json"]))
raw_cats = sorted(raw_train.loadCats(raw_train.getCatIds()), key=lambda x: x["id"])

print("\nCategorías COCO (originales):", [(c["id"], c["name"]) for c in raw_cats])
print(f"Total imágenes de entrenamiento: {len(raw_train.getImgIds())}")
print(f"Número de anotaciones train: {len(raw_train.getAnnIds())}")

En principio vemos que existen 6 categorías distintas: *'lettuce', 'Ready', 'empty_pod', 'germination', 'pod'* y *'young'*, repartidas entre 1060 imágenes de entrenamiento y con un total de 13812 anotaciones.

Mostramos las anotaciones por categoría.

In [ ]:
for c in raw_cats:
    n = len(raw_train.getAnnIds(catIds=[c["id"]]))
    print(f"  id {c['id']}: {c['name']:11s} -> {n} anotaciones")

Comprobamos que todas las categorías tienen anotaciones menos `lettuce`.

**Paso:** limpiar las anotaciones y crear un JSON limpio por split.

**Decisiones y justificación:**
1. **Eliminar la categoría `lettuce` (id 0).** En Roboflow es la categoría "padre" del dataset y no tiene anotaciones propias. Si se dejara, el modelo
   tendría 6 clases en vez de las 5 reales del guion, con una clase "fantasma" que nunca aparece.
2. **Renumerar las clases a 1…5.** En torchvision la etiqueta 0 es el fondo; con ids contiguos 1…5 la etiqueta del modelo **coincide** con el `category_id`
   y no hace falta ninguna conversión (la versión anterior restaba 1 al evaluar y desplazaba todas las clases).
3. **Recortar las cajas a la imagen y descartar las degeneradas** (ancho o alto < 1 px): una caja de tamaño nulo provoca errores y pérdidas `NaN` al entrenar.
4. **Se conservan las imágenes sin objetos** (en el JSON): se usan en la evaluación; Detectron2 las ignora solo al entrenar.

In [ ]:
def build_clean_coco(raw, out_dir):
    """Genera un JSON COCO limpio por split. Devuelve (CLEAN, class_names, class_ids)."""
    # Creamos el directorio de salida si no existe
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Leemos el JSON de entrenamiento original y obtenemos los nombres de las categorías y los IDs utilizados
    train = json.loads(Path(raw["train"]["json"]).read_text())
    old_names = {c["id"]: c["name"] for c in train["categories"]}
    used_ids = sorted({a["category_id"] for a in train["annotations"]})
    dropped = [n for i, n in old_names.items() if i not in used_ids] # categorías descartadas
    print("Categorías descartadas (sin anotaciones en train):", dropped)

    # Creamos un diccionario que mapea los nombres de las categorías a nuevos IDs consecutivos (1..K) para las categorías utilizadas
    name2new = {old_names[i]: n + 1 for n, i in enumerate(used_ids)}

    # Iteramos sobre cada split y generamos un nuevo JSON COCO limpio
    # Descartamos anotaciones de categorías no utilizadas y cajas degeneradas
    clean = {}
    for split in SPLITS:
        d = json.loads(Path(raw[split]["json"]).read_text())
        id2name = {c["id"]: c["name"] for c in d["categories"]}
        imgs = {im["id"]: im for im in d["images"]}
        new_anns, stats = [], Counter()
        
        for a in d["annotations"]:
            name = id2name.get(a["category_id"])
            
            # Descartamos cajas degeneradas (cajas con ancho o alto menor que 1 píxel)
            im = imgs[a["image_id"]]; x, y, w, h = a["bbox"]
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(im["width"]), x + w), min(float(im["height"]), y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                stats["caja_degenerada"] += 1
                continue
            
            # Creamos la nueva anotación con el nuevo ID de categoría y la caja ajustada
            new_anns.append({"id": len(new_anns) + 1, "image_id": a["image_id"],
                             "category_id": name2new[name], "bbox": [x1, y1, x2 - x1, y2 - y1],
                             "area": (x2 - x1) * (y2 - y1), "iscrowd": 0})
        
        # Guardamos el nuevo JSON COCO limpio para el split actual
        out = {"images": d["images"], "annotations": new_anns,
               "categories": [{"id": i, "name": n} for n, i in name2new.items()]}
        
        # Guardamos el JSON limpio en el directorio de salida y almacenamos la información del split en el diccionario 'clean'
        p = out_dir / f"{split}_clean.json"; p.write_text(json.dumps(out))
        clean[split] = {"json": p, "images": Path(raw[split]["images"])}
        print(f"{split:5s}: {len(d['images'])} imgs | {len(d['annotations'])} -> {len(new_anns)} anotaciones | descartes: {dict(stats)}")
    names = list(name2new.keys())
    ids = list(name2new.values())
    return clean, names, ids

CLEAN, CLASS_NAMES, CLASS_IDS = build_clean_coco(RAW, CLEAN_DIR)
K = len(CLASS_NAMES) # nº de clases reales
# Creamos un diccionario que mapea los IDs de las clases a sus índices
CID2IDX = {c: i for i, c in enumerate(CLASS_IDS)}
print(f"\nK = {K} clases:", dict(zip(CLASS_IDS, CLASS_NAMES)))

# Creamos un diccionario que mapea los splits a sus objetos COCO correspondientes para facilitar la evaluación
COCO_GT = {s: COCO(str(CLEAN[s]["json"])) for s in SPLITS}

## 3. Análisis exploratorio del dataset

**Paso:** medir las características que condicionan el diseño de las mejoras: reparto de clases, nº de objetos por imagen y **tamaño de los objetos**.
Es lo que pide el guion en la exposición (punto 1) y lo que **justifica** las decisiones de la fase de mejora.

**Qué buscamos y por qué importa:**
- *Desbalance de clases* → si hay clases raras, plantear muestreo balanceado o ponderar la pérdida.
- *Tamaño de objeto* → SSD300 reescala **todas** las imágenes a 300×300; los objetos pequeños pierden detalle. Calculamos el tamaño efectivo tras el reescalado.
- *Vista cenital* → no hay "arriba/abajo" privilegiado: los giros y volteos de la imagen producen escenas igual de realistas (justifica el aumento geométrico).

In [ ]:
# Creamos un DataFrame con las anotaciones de todos los splits
# Incluimos información sobre el tamaño y la relación de aspecto de cada objeto
def annotations_dataframe(coco, split):
    rows = []
    for a in coco.anns.values():
        im = coco.imgs[a["image_id"]]

        # Calculamos el área de la caja delimitadora
        x, y, w, h = a["bbox"]
        area = w * h

        # Agregamos la información al DataFrame
        rows.append({"split": split, "image_id": a["image_id"],
                     "cls": CLASS_NAMES[CID2IDX[a["category_id"]]], # nombre de la clase
                     "w": w, "h": h, "area": area, # área de la caja delimitadora
                     "sqrt_area": math.sqrt(area), # raíz cuadrada del área de la caja delimitadora
                     "rel_size": math.sqrt(area / (im["width"] * im["height"])), # tamaño relativo del objeto respecto al tamaño de la imagen
                     "aspect": w / h}) # relación entre el ancho y el alto de la caja delimitadora
    df = pd.DataFrame(rows)
    # Clasificamos los objetos en categorías de tamaño: pequeño, mediano y grande según el área de la caja delimitadora
    # Los límites de las categorías se establecen en 32^2 y 96^2 píxeles cuadrados, según la convención de COCO
    df["size_cat"] = pd.cut(df["area"], [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])
    return df

# Creamos el DataFrame con las anotaciones para todos los splits
ANN_DF = pd.concat([annotations_dataframe(COCO_GT[s], s) for s in SPLITS], ignore_index=True)

# Mostramos un resumen de las imágenes y anotaciones por split
print("Imágenes y anotaciones por split:")
summary = []
for s in SPLITS:
    c = COCO_GT[s]
    per_img = Counter(a["image_id"] for a in c.anns.values()) # contamos el número de anotaciones por imagen
    n_img = len(c.imgs)
    n_empty = sum(1 for i in c.imgs if per_img.get(i, 0) == 0) # contamos el número de imágenes sin anotaciones
    summary.append({"split": s, "imágenes": n_img, "anotaciones": len(c.anns), "imgs_sin_objetos": n_empty,
                    "objs/img media": round(len(c.anns) / n_img, 1), # número medio de objetos por imagen
                    "objs/img máx": max(per_img.values())}) # número máximo de objetos en una imagen
display(pd.DataFrame(summary).set_index("split"))

# Mostramos un resumen de las anotaciones por clase y split
print("\nAnotaciones por clase y split:")
class_counts = ANN_DF.pivot_table(index="cls", columns="split", values="area", aggfunc="count").reindex(CLASS_NAMES)[list(SPLITS)]
class_counts["% train"] = (100 * class_counts["train"] / class_counts["train"].sum()).round(1)
display(class_counts)

# Mostramos un resumen de la resolución de las imágenes en el dataset
print("\nResolución de las imágenes:")
display(pd.Series([(im["width"], im["height"]) for c in COCO_GT.values() for im in c.imgs.values()]).value_counts().head())

# SSD300 redimensiona todas las imágenes a 300x300
# Calculamos el tamaño efectivo del objeto en píxeles tras el reescalado a 300x300 píxeles
ANN_DF["px_ssd300"] = ANN_DF["rel_size"] * 300
tr = ANN_DF[ANN_DF.split == "train"]

# Mostramos un resumen de la distribución del tamaño de los objetos en el conjunto de entrenamiento
print("\nTamaño de objeto (train):", tr["size_cat"].value_counts(normalize=True).round(3).to_dict())

# Mostramos un resumen de la raíz cuadrada del área de los objetos en píxeles originales y sus cuantiles
print("sqrt(área) px originales, cuantiles:", tr["sqrt_area"].quantile([.05, .25, .5, .75, .95]).round(1).to_dict())

# Mostramos un resumen del porcentaje de objetos con tamaño efectivo menor a 16 píxeles en SSD300
# Los objetos muy pequeños pueden ser difíciles de detectar para el modelo SSD300
# Este porcentaje nos da una idea de cuántos objetos podrían ser problemáticos en términos de detección
print(f"% objetos con tamaño efectivo < 16 px en SSD300: {100*(tr['px_ssd300'] < 16).mean():.1f}%")

Mostramos un resumen gráfico de las anotaciones por clase, el número de objetos por imagen, la distribución del tamaño de los objetos y la relación de aspecto de las cajas delimitadoras.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(22, 4.2))

# Mostramos un gráfico de barras con el número de anotaciones por clase para cada split
class_counts[list(SPLITS)].plot.bar(ax=axs[0]); axs[0].set_title("Anotaciones por clase"); axs[0].set_xlabel("")

# Mostramos un histograma con el número de objetos por imagen para cada split
for s in SPLITS:
    cnt = Counter(a["image_id"] for a in COCO_GT[s].anns.values())
    axs[1].hist([cnt.get(i, 0) for i in COCO_GT[s].imgs], bins=30, alpha=.5, label=s)
axs[1].set_title("Objetos por imagen")
axs[1].legend()

# Mostramos un histograma que refleje la distribución del tamaño de los objetos
axs[2].hist(tr["sqrt_area"], bins=60)
axs[2].set_xscale("log")
axs[2].set_title("√área de la caja (px originales)")

# Mostramos un histograma con la relación de aspecto (ancho/alto) de los objetos por clase
for cls in CLASS_NAMES:
    axs[3].hist(tr[tr.cls == cls]["aspect"].clip(0, 4), bins=40, alpha=.5, label=cls)
axs[3].set_title("Relación de aspecto (w/h)")
axs[3].legend(fontsize=7)

plt.tight_layout()
plt.savefig(WORK_DIR / "figs" / "eda_overview.png", dpi=130)
plt.show()

# Mostramos el tamaño medio por clase para ver si hay clases sistemáticamente más pequeñas
# Esto puede ser relevante para la detección, ya que las clases más pequeñas pueden ser más difíciles de detectar para el modelo
display(tr.groupby("cls")[["sqrt_area", "px_ssd300"]].agg(["median", "min", "max"]).round(1).reindex(CLASS_NAMES))

Lo siguiente que haremos será mostrar algunas de las imágenes de entrenamiento con sus respectivos tamaños y anotaciones.

In [ ]:
# Creamos un array de colores para cada clase utilizando el colormap 'tab10' de Matplotlib
CLASS_COLORS = plt.cm.tab10(np.arange(K))

def draw_gt(coco, image_dir, image_ids, ncols=3):
    nrows = math.ceil(len(image_ids) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows)); axs = np.atleast_1d(axs).ravel()
    for ax in axs: ax.axis("off")
    for ax, iid in zip(axs, image_ids):
        info = coco.imgs[iid]
        ax.imshow(Image.open(Path(image_dir) / info["file_name"]).convert("RGB"))
        n_obj = 0
        for a in coco.loadAnns(coco.getAnnIds(imgIds=iid)):
            x, y, w, h = a["bbox"]
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, ec=CLASS_COLORS[CID2IDX[a["category_id"]]]))
            n_obj += 1
        ax.set_title(f"({info['width']}x{info['height']}) \n{n_obj} objetos detectados")
    handles = [patches.Patch(color=CLASS_COLORS[i], label=n) for i, n in enumerate(CLASS_NAMES)]
    fig.legend(handles=handles, loc="lower center", ncol=K)
    plt.tight_layout(); plt.show()

rng = random.Random(SEED)
draw_gt(COCO_GT["train"], CLEAN["train"]["images"], rng.sample(sorted(COCO_GT["train"].imgs), 3))

### ✏️ Completar con las cifras de las celdas anteriores (1 diapositiva)
- Clase más/menos frecuente y su % en train: _…_
- % de objetos *small/medium/large* y % por debajo de 16 px en SSD300: _…_
- Objetos por imagen (media/máximo) y si `MAX_DETS=300` es suficiente: _…_
- Retos visuales observados en las imágenes (oclusión, parecido entre clases, iluminación…): _…_

## 4. Infraestructura común: dataset, evaluación, tiempos y resultados

Partimos de los JSON **limpios** (5 clases, ids 1…5, cajas recortadas). Como los `category_id` son contiguos desde 1, la etiqueta que usa torchvision (0 = fondo) **coincide** con el `category_id` y no hace falta ninguna conversión.
Creamos el `Dataset` COCO para torchvision (SSD) y los `DataLoader`s; Detectron2 leerá los mismos JSON directamente.

In [ ]:
assert CLASS_IDS == list(range(1, K + 1)), "Los ids deben ser 1..K para que label == category_id"

class CocoDetection(Dataset):
    """Dataset COCO para torchvision. Devuelve (imagen [0,1] CxHxW, target con cajas xyxy y etiquetas 1..K)."""
    def __init__(self, coco, image_dir, train=False):
        self.coco = coco
        self.image_dir = Path(image_dir)
        self.ids = sorted(coco.getImgIds())
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]
        info = self.coco.imgs[img_id]
        image = Image.open(self.image_dir / info["file_name"]).convert("RGB")
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))

        boxes  = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]
        labels = [a["category_id"] for a in anns]

        image = TF.to_tensor(image)
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),  # reshape: imágenes sin objetos
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([img_id]),
        }

        if self.train:
            pass  # aquí se aplicaría el aumento de datos (fase de mejoras); el baseline no usa aumento

        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Un Dataset por split, reutilizando los objetos COCO de ground truth ya cargados
DS = {s: CocoDetection(COCO_GT[s], CLEAN[s]["images"], train=(s == "train")) for s in SPLITS}

BATCH_SSD = 4
NUM_WORKERS = 2
train_loader = DataLoader(DS["train"], batch_size=BATCH_SSD, shuffle=True, collate_fn=collate_fn,
                          num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=make_generator())
val_loader = DataLoader(DS["val"], batch_size=BATCH_SSD, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)
print({s: len(DS[s]) for s in SPLITS})

Definimos la evaluación **común** a ambos detectores: recibe predicciones en formato COCO y calcula mAP@[.5:.95], mAP50, mAP75, mAP por tamaño y AP por clase con `COCOeval` (`maxDets = [1, 10, MAX_DETS]`).
También un registro `results.csv` donde se acumulan todos los experimentos (precisión, tiempo de entrenamiento y tiempo de inferencia por imagen).

In [ ]:
METRIC_NAMES = ["mAP", "mAP50", "mAP75", "mAP_small", "mAP_medium", "mAP_large"]

def evaluate_coco(coco_gt, predictions, max_dets=MAX_DETS):
    """Evaluación COCO (bbox) idéntica para SSD y Cascade R-CNN."""
    if not predictions:
        return {**{n: 0.0 for n in METRIC_NAMES}, **{f"AP_{c}": 0.0 for c in CLASS_NAMES}}
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(predictions)
        ev = COCOeval(coco_gt, coco_dt, "bbox")
        ev.params.maxDets = [1, 10, max_dets]
        ev.evaluate(); ev.accumulate(); ev.summarize()
    m = {n: float(ev.stats[i]) for i, n in enumerate(METRIC_NAMES)}
    prec = ev.eval["precision"]                      # [T, R, K, A, M]; A=0 -> todas las áreas, M=2 -> max_dets
    for k, cname in enumerate(CLASS_NAMES):          # catIds de COCOeval van ordenados = CLASS_IDS
        p = prec[:, :, k, 0, 2]; p = p[p > -1]
        m[f"AP_{cname}"] = float(p.mean()) if p.size else float("nan")
    return m

RESULTS_CSV = WORK_DIR / "results.csv"

def log_result(exp, detector, split, metrics, train_time_s=float("nan"), infer_ms_img=float("nan")):
    """Añade (o sustituye) una fila en results.csv y devuelve la tabla completa."""
    row = {"exp": exp, "detector": detector, "split": split, **metrics,
           "train_min": round(train_time_s / 60, 1), "infer_ms_img": round(infer_ms_img, 1)}
    df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
    if len(df):
        df = df[~((df["exp"] == exp) & (df["split"] == split))]
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(RESULTS_CSV, index=False)
    return df

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

## 5. Baseline SSD300 (torchvision)

Construimos el detector SSD300 con VGG16 como backbone, con pesos preentrenados en COCO, y sustituimos la cabeza de clasificación por una nueva para nuestras `K` clases (+1 de fondo).
Los umbrales de inferencia (`score_thresh`, `nms_thresh`, `detections_per_img`) son los comunes definidos en la configuración.

In [ ]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead

def build_ssd(pretrained=True):
    weights = SSD300_VGG16_Weights.COCO_V1 if pretrained else None
    model = ssd300_vgg16(weights=weights, score_thresh=SCORE_THRESH_EVAL,
                         nms_thresh=NMS_IOU, detections_per_img=MAX_DETS)

    # Canales de entrada de cada nivel de características y nº de anchors por posición
    in_channels = [layer.in_channels for layer in model.head.classification_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()

    # Sustituimos la cabeza de clasificación para nuestras clases (+1 para fondo)
    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors, num_classes=K + 1)
    return model

seed_everything()
ssd = build_ssd(PRETRAINED).to(DEVICE)
print("SSD300 preparado correctamente.")

Entrenamiento del baseline: SGD (lr 0.002, momentum 0.9, weight decay 5e-4), `StepLR` (÷10 cada 5 épocas), 15 épocas, *clipping* de gradiente a 10 y **sin aumento de datos**.
Guardamos el checkpoint con menor pérdida en *val*. Si ya existe un checkpoint en `work/ckpt`, se reutiliza.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train(); losses = []
    for images, targets in loader:
        images = [im.to(device) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def mean_detection_loss(model, loader, device):
    # Torchvision solo calcula la loss en modo train; no se actualizan gradientes.
    model.train(); vals = []
    for images, targets in loader:
        images = [im.to(device) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        vals.append(float(sum(model(images, targets).values()).cpu()))
    model.eval()
    return float(np.mean(vals))

EPOCHS_SSD = 15
LR_SSD = 0.002
SSD_CKPT = WORK_DIR / "ckpt" / "ssd300_baseline.pth"
SSD_HIST = WORK_DIR / "ckpt" / "ssd300_baseline_history.json"

if SSD_CKPT.exists() and SSD_HIST.exists():
    info = json.loads(SSD_HIST.read_text())
    history_ssd, ssd_train_time = info["history"], info["train_time_s"]
    ssd.load_state_dict(torch.load(SSD_CKPT, map_location=DEVICE))
    print(f"Checkpoint SSD reutilizado ({SSD_CKPT})")
else:
    optimizer = torch.optim.SGD([p for p in ssd.parameters() if p.requires_grad],
                                lr=LR_SSD, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    history_ssd, best_val = [], float("inf")
    sync(); t0 = time.time()
    for epoch in range(EPOCHS_SSD):
        tr = train_one_epoch(ssd, train_loader, optimizer, DEVICE)
        vl = mean_detection_loss(ssd, val_loader, DEVICE)
        scheduler.step()
        history_ssd.append({"epoch": epoch + 1, "train_loss": tr, "val_loss": vl})
        print(f"SSD epoch {epoch+1}/{EPOCHS_SSD}: train={tr:.4f} val={vl:.4f}")
        if vl < best_val:
            best_val = vl
            torch.save(ssd.state_dict(), SSD_CKPT)
    sync(); ssd_train_time = time.time() - t0
    SSD_HIST.write_text(json.dumps({"history": history_ssd, "train_time_s": ssd_train_time}))
    ssd.load_state_dict(torch.load(SSD_CKPT, map_location=DEVICE))   # nos quedamos con el mejor en val

ssd.eval()
pd.DataFrame(history_ssd).plot(x="epoch", y=["train_loss", "val_loss"], grid=True, title="SSD300 baseline")
plt.savefig(WORK_DIR / "figs" / "ssd_baseline_loss.png", dpi=130)
plt.show()
print(f"Tiempo de entrenamiento: {ssd_train_time/60:.1f} min")

Evaluamos el SSD300 en *val* y *test* con la evaluación común. El tiempo de inferencia por imagen incluye la carga de datos (igual que en Cascade R-CNN).

In [ ]:
@torch.no_grad()
def predict_ssd(model, dataset, batch_size=4):
    """Predicciones en formato COCO y tiempo medio (ms/imagen)."""
    model.eval(); preds = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    sync(); t0 = time.time()
    for images, targets in loader:
        outputs = model([im.to(DEVICE) for im in images])
        for target, out in zip(targets, outputs):
            image_id = int(target["image_id"].item())
            for (x1, y1, x2, y2), score, label in zip(out["boxes"].cpu().tolist(),
                                                      out["scores"].cpu().tolist(),
                                                      out["labels"].cpu().tolist()):
                preds.append({"image_id": image_id, "category_id": CLASS_IDS[int(label) - 1],   # label 1..K
                              "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})
    sync()
    return preds, 1000 * (time.time() - t0) / len(dataset)

for split in ("val", "test"):
    preds, ms = predict_ssd(ssd, DS[split])
    (WORK_DIR / "preds" / f"ssd300_baseline_{split}.json").write_text(json.dumps(preds))
    m = evaluate_coco(COCO_GT[split], preds)
    log_result("SSD300 baseline", "SSD300", split, m, ssd_train_time, ms)
    print(f"SSD300 [{split}] mAP={m['mAP']:.3f} mAP50={m['mAP50']:.3f} mAP75={m['mAP75']:.3f} | {ms:.1f} ms/img")

# Liberamos la GPU antes de entrenar Cascade R-CNN
del ssd; gc.collect(); torch.cuda.empty_cache()

## 6. Baseline Cascade R-CNN (Detectron2)

Registramos los JSON limpios en Detectron2 y configuramos Cascade R-CNN (R-50-FPN, 3x) con pesos de COCO, sin cabeza de máscaras (`MASK_ON = False`), 5 clases y los mismos umbrales de inferencia que SSD.
Hiperparámetros del baseline: `IMS_PER_BATCH = 2`, `BASE_LR = 0.00025`, `MAX_ITER = max(500, N_train/2 · 10)` (≈10 épocas), decaimiento ×0.1 al 70 % y 90 %.

Usamos un `Trainer` propio con `build_evaluator`, necesario para que la evaluación periódica en *val* (`TEST.EVAL_PERIOD`) funcione con `DefaultTrainer`.

In [ ]:
# Registro de los splits limpios en Detectron2
for split in SPLITS:
    name = f"lettuce_{split}"
    if name in DatasetCatalog.list(): DatasetCatalog.remove(name)
    register_coco_instances(name, {}, str(CLEAN[split]["json"]), str(CLEAN[split]["images"]))
    MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)

CASCADE_CONFIG = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
CASCADE_DIR = WORK_DIR / "d2" / "cascade_baseline"
CASCADE_DIR.mkdir(parents=True, exist_ok=True)

def make_cascade_cfg(output_dir):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(CASCADE_CONFIG))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CASCADE_CONFIG) if PRETRAINED else ""
    cfg.MODEL.MASK_ON = False
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = K
    cfg.DATASETS.TRAIN = ("lettuce_train",)
    cfg.DATASETS.TEST = ("lettuce_val",)
    cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
    cfg.SEED = SEED
    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = 0.00025
    cfg.SOLVER.MAX_ITER = max(500, len(DS["train"]) // 2 * 10)
    cfg.SOLVER.STEPS = (int(cfg.SOLVER.MAX_ITER * 0.7), int(cfg.SOLVER.MAX_ITER * 0.9))
    cfg.SOLVER.CHECKPOINT_PERIOD = max(100, cfg.SOLVER.MAX_ITER // 3)
    cfg.TEST.EVAL_PERIOD = max(100, cfg.SOLVER.MAX_ITER // 5)
    # Umbrales de inferencia comunes con SSD
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_EVAL
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = NMS_IOU
    cfg.TEST.DETECTIONS_PER_IMAGE = MAX_DETS
    cfg.OUTPUT_DIR = str(output_dir)
    return cfg

class CascadeTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name):
        return COCOEvaluator(dataset_name, distributed=False, output_dir=os.path.join(cfg.OUTPUT_DIR, "eval"))

cfg = make_cascade_cfg(CASCADE_DIR)
CASCADE_FINAL = CASCADE_DIR / "model_final.pth"
CASCADE_TIME = CASCADE_DIR / "train_time.json"
print(f"MAX_ITER={cfg.SOLVER.MAX_ITER} | STEPS={cfg.SOLVER.STEPS} | EVAL_PERIOD={cfg.TEST.EVAL_PERIOD}")

In [ ]:
if CASCADE_FINAL.exists() and CASCADE_TIME.exists():
    cascade_train_time = json.loads(CASCADE_TIME.read_text())["train_time_s"]
    print(f"Checkpoint Cascade reutilizado ({CASCADE_FINAL})")
else:
    seed_everything()
    trainer = CascadeTrainer(cfg)
    trainer.resume_or_load(resume=True)   # reanuda desde el último checkpoint si la sesión se interrumpió
    sync(); t0 = time.time()
    trainer.train()
    sync(); cascade_train_time = time.time() - t0   # (si se reanudó, solo cuenta el tiempo de esta sesión)
    CASCADE_TIME.write_text(json.dumps({"train_time_s": cascade_train_time}))
    del trainer; gc.collect(); torch.cuda.empty_cache()
print(f"Tiempo de entrenamiento: {cascade_train_time/60:.1f} min")

In [ ]:
# Curva de pérdida de entrenamiento (metrics.json lo escribe Detectron2)
mj = CASCADE_DIR / "metrics.json"
if mj.exists():
    rows = [json.loads(l) for l in mj.read_text().splitlines() if l.strip()]
    it = [r["iteration"] for r in rows if "total_loss" in r]; tl = [r["total_loss"] for r in rows if "total_loss" in r]
    plt.plot(it, tl); plt.grid(True); plt.xlabel("iteración"); plt.title("Cascade R-CNN baseline: total_loss")
    plt.savefig(WORK_DIR / "figs" / "cascade_baseline_loss.png", dpi=130); plt.show()

Evaluamos Cascade R-CNN con el **mismo** `evaluate_coco` que SSD: generamos las predicciones en formato COCO con `DefaultPredictor` (imágenes leídas en BGR, como espera Detectron2) y las pasamos por la evaluación común.

In [ ]:
def predict_d2(predictor, coco_gt, image_dir):
    """Predicciones en formato COCO y tiempo medio (ms/imagen) de un DefaultPredictor."""
    preds = []
    sync(); t0 = time.time()
    for iid in sorted(coco_gt.imgs):
        img = cv2.imread(str(Path(image_dir) / coco_gt.imgs[iid]["file_name"]))   # BGR
        inst = predictor(img)["instances"].to("cpu")
        boxes = inst.pred_boxes.tensor.numpy(); scores = inst.scores.numpy(); classes = inst.pred_classes.numpy()
        for (x1, y1, x2, y2), s, c in zip(boxes, scores, classes):
            preds.append({"image_id": int(iid), "category_id": CLASS_IDS[int(c)],
                          "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    sync()
    return preds, 1000 * (time.time() - t0) / len(coco_gt.imgs)

cfg.MODEL.WEIGHTS = str(CASCADE_FINAL)
predictor = DefaultPredictor(cfg)

for split in ("val", "test"):
    preds, ms = predict_d2(predictor, COCO_GT[split], CLEAN[split]["images"])
    (WORK_DIR / "preds" / f"cascade_baseline_{split}.json").write_text(json.dumps(preds))
    m = evaluate_coco(COCO_GT[split], preds)
    log_result("Cascade R-CNN baseline", "Cascade R-CNN", split, m, cascade_train_time, ms)
    print(f"Cascade R-CNN [{split}] mAP={m['mAP']:.3f} mAP50={m['mAP50']:.3f} mAP75={m['mAP75']:.3f} | {ms:.1f} ms/img")

## 7. Resultados del baseline

Tabla acumulada (`results.csv`) y comparación SSD300 vs Cascade R-CNN. Las decisiones se toman con *val*; *test* es la cifra a reportar.

In [ ]:
res = pd.read_csv(RESULTS_CSV)
cols = ["exp", "split"] + METRIC_NAMES + ["train_min", "infer_ms_img"]
display(res[cols].round(3).sort_values(["split", "exp"]))

print("\nAP por clase:")
display(res[["exp", "split"] + [f"AP_{c}" for c in CLASS_NAMES]].round(3).sort_values(["split", "exp"]))

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, split in zip(axs, ("val", "test")):
    r = res[res.split == split].set_index("exp")[["mAP50", "mAP"]]
    r.plot.bar(ax=ax, rot=0); ax.set_title(f"SSD300 vs Cascade R-CNN ({split})"); ax.set_xlabel(""); ax.grid(axis="y")
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "baseline_comparison.png", dpi=130); plt.show()